# Baseline Comparison (Updated)
Compares the trivial `constant_mean` baseline, lexical TF-IDF, and sentence-embedding baselines with comprehensive bootstrap 95% CIs across all metrics, and computes paired bootstrap differences.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression

os.environ['HF_HOME'] = r'D:\Projects 2026\genai-evaluation-and-assessment-system\.cache\huggingface'
sys.path.insert(0, '../')
from src.embeddings import cosine_scores
from src.metrics import evaluate, paired_bootstrap_diff, bootstrap_ci

df_dev  = pd.read_csv('../data/processed/dev.csv')
df_test = pd.read_csv('../data/processed/test.csv')
preds_tfidf = pd.read_csv('../results/preds_tfidf.csv')
print(f'Test rows: {len(df_test)}, Dev rows: {len(df_dev)}')

## 1. Predictions Setup
1. **constant_mean**: Predicts the DEV set mean score for every TEST response.
2. **TF-IDF (Calibrated)**: OLS linear regression on DEV cosine scores -> TEST.
3. **MiniLM & MPNet (Calibrated)**: OLS linear regression on DEV cosine scores -> TEST, clipped to [0, 5].

In [ ]:
dev_mean = float(df_dev['human_score'].mean())
human = df_test['human_score'].values

test_preds = {
    'constant_mean': np.full(len(df_test), dev_mean),
    'TF-IDF (Calibrated)': preds_tfidf['pred_calibrated'].values
}

MODELS = {
    'MiniLM-L6-v2': 'sentence-transformers/all-MiniLM-L6-v2',
    'MPNet-base-v2': 'sentence-transformers/all-mpnet-base-v2',
}

for short, full in MODELS.items():
    dev_cos  = cosine_scores(full, df_dev['reference_answer'].tolist(), df_dev['student_answer'].tolist())
    test_cos = cosine_scores(full, df_test['reference_answer'].tolist(), df_test['student_answer'].tolist())
    lr = LinearRegression().fit(dev_cos.reshape(-1,1), df_dev['human_score'])
    test_preds[f'{short} (Calibrated)'] = np.clip(lr.predict(test_cos.reshape(-1,1)), 0, 5)
    print(f'{short}: intercept={lr.intercept_:.3f}, coef={lr.coef_[0]:.3f}')

## 2. Bootstrap 95% CIs (1,000 resamples, seed=42)
Computed for MAE, RMSE, Pearson, Spearman, and QWK across all evaluated methods.

In [ ]:
METRICS = ['mae', 'rmse', 'pearson', 'spearman', 'qwk']
METRIC_COLS = {
    'mae': 'MAE 95% CI',
    'rmse': 'RMSE 95% CI',
    'pearson': 'Pearson 95% CI',
    'spearman': 'Spearman 95% CI',
    'qwk': 'QWK 95% CI'
}

ci_results = {}
for method, pred in test_preds.items():
    ci_results[method] = {}
    for m in METRICS:
        lo, hi = bootstrap_ci(human, pred, metric=m, n=1000, seed=42)
        ci_results[method][m] = (lo, hi)
        print(f'[{method}] {m.upper()}: [{lo:.3f}, {hi:.3f}]' if not np.isnan(lo) else f'[{method}] {m.upper()}: —')

## 3. Results Table with Extended CIs

In [ ]:
rows = []
for method, pred in test_preds.items():
    m = evaluate(human, pred)
    row = {
        'Method': method,
        'MAE': m['mae'],
        'RMSE': m['rmse'],
        'Pearson': m['pearson'],
        'Spearman': m['spearman'],
        'QWK': m['qwk'],
        'Within±1': m['within_1']
    }
    for m_key, col_name in METRIC_COLS.items():
        lo, hi = ci_results[method][m_key]
        row[col_name] = f'[{lo:.3f}, {hi:.3f}]' if not np.isnan(lo) else '—'
    rows.append(row)

# Append scaled baselines from main_results.csv
main_df = pd.read_csv('../results/main_results.csv').set_index('Method')
for meth in [
    'TF-IDF (Scaled)',
    'MiniLM-L6-v2 (Scaled)',
    'sentence-transformers/all-MiniLM-L6-v2 (Scaled)',
    'MPNet-base-v2 (Scaled)',
    'sentence-transformers/all-mpnet-base-v2 (Scaled)'
]:
    if meth in main_df.index:
        r = main_df.loc[meth]
        row = {
            'Method': meth,
            'MAE': r['mae'],
            'RMSE': r['rmse'],
            'Pearson': r['pearson'],
            'Spearman': r['spearman'],
            'QWK': r['qwk'],
            'Within±1': r['within_1']
        }
        for m_key, col_name in METRIC_COLS.items():
            row[col_name] = '—'
        rows.append(row)

table_df = pd.DataFrame(rows).drop_duplicates(subset=['Method']).set_index('Method')
table_df.to_csv('../results/baseline_table.csv')
print('Saved table to ../results/baseline_table.csv')
table_df

## 4. Paired Bootstrap Hypothesis Testing
Using `paired_bootstrap_diff` (1,000 resamples, seed=42) on identical resampled indices to test:
1. Each calibrated baseline against `constant_mean`.
2. `MiniLM` against `TF-IDF`.

In [ ]:
paired_records = []

# Calibrated baselines vs constant_mean
for name in ['TF-IDF (Calibrated)', 'MiniLM-L6-v2 (Calibrated)', 'MPNet-base-v2 (Calibrated)']:
    for m in ['mae', 'rmse', 'qwk']:
        res = paired_bootstrap_diff(human, test_preds[name], test_preds['constant_mean'], metric=m, n=1000, seed=42)
        diff, (ci_lo, ci_hi) = res
        paired_records.append({
            'Comparison': f'{name} vs constant_mean',
            'Metric': m.upper(),
            'Mean Difference': diff,
            '95% CI Lower': ci_lo,
            '95% CI Upper': ci_hi,
            'Significant (p<0.05)': (ci_lo > 0 and ci_hi > 0) or (ci_lo < 0 and ci_hi < 0)
        })

# MiniLM vs TF-IDF
for m in ['mae', 'rmse', 'pearson', 'spearman', 'qwk']:
    res = paired_bootstrap_diff(human, test_preds['MiniLM-L6-v2 (Calibrated)'], test_preds['TF-IDF (Calibrated)'], metric=m, n=1000, seed=42)
    diff, (ci_lo, ci_hi) = res
    paired_records.append({
        'Comparison': 'MiniLM-L6-v2 vs TF-IDF',
        'Metric': m.upper(),
        'Mean Difference': diff,
        '95% CI Lower': ci_lo,
        '95% CI Upper': ci_hi,
        'Significant (p<0.05)': (ci_lo > 0 and ci_hi > 0) or (ci_lo < 0 and ci_hi < 0)
    })

paired_df = pd.DataFrame(paired_records)
paired_df.to_csv('../results/paired_bootstrap_diffs.csv', index=False)
print('Saved paired differences to ../results/paired_bootstrap_diffs.csv')
paired_df

## 5. Grouped Bar Chart of Baselines

In [ ]:
plot_methods = ['constant_mean', 'TF-IDF (Calibrated)', 'MiniLM-L6-v2 (Calibrated)', 'MPNet-base-v2 (Calibrated)']
plot_df = table_df.loc[plot_methods]
x, width = np.arange(len(plot_methods)), 0.35

fig, ax = plt.subplots(figsize=(11, 5))
rects1 = ax.bar(x - width/2, plot_df['MAE'], width, label='MAE', color='#4C72B0', alpha=0.85)
rects2 = ax.bar(x + width/2, plot_df['QWK'], width, label='QWK', color='#DD8452', alpha=0.85)

mae_lo = [plot_df.loc[m, 'MAE'] - ci_results[m]['mae'][0] for m in plot_methods]
mae_hi = [ci_results[m]['mae'][1] - plot_df.loc[m, 'MAE'] for m in plot_methods]
ax.errorbar(x - width/2, plot_df['MAE'], yerr=[mae_lo, mae_hi],
            fmt='none', color='black', capsize=4, linewidth=1.2)

short_labels = [m.replace(' (Calibrated)', '\n(Calibrated)').replace('constant_mean', 'constant\nmean') for m in plot_methods]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Baseline Evaluation — MAE & QWK with 95% Bootstrap CIs', fontsize=12)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_ylim(0, max(plot_df['MAE'].max(), plot_df['QWK'].max()) * 1.25)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/baseline_comparison.png', dpi=150)
plt.show()

## 6. Summary of Key Findings

1. **`constant_mean` Deception**:
   - The trivial dev-mean baseline achieves an MAE of **0.926** (and 78.7% within $\pm 1$) solely due to positive skew in student scores.
   - However, its QWK is strictly **0.000**, and correlation is undefined.

2. **Calibrated Baselines vs `constant_mean` (Paired Test)**:
   - **MiniLM vs constant_mean**: Statistically significant reduction in MAE (diff = **-0.1450**, 95% CI `[-0.2505, -0.0435]`) and RMSE (diff = **-0.1670**, 95% CI `[-0.2797, -0.0637]`), with QWK improving to **+0.4000** (95% CI `[+0.2647, +0.5284]`).
   - **TF-IDF vs constant_mean**: Fails to significantly beat constant_mean on MAE (diff = **-0.0560**, 95% CI `[-0.1705, +0.0638]`, includes 0) and RMSE (diff = **+0.0243**, 95% CI `[-0.0988, +0.1340]`), though it obtains a small positive QWK of 0.163.

3. **MiniLM vs TF-IDF (Paired Test)**:
   - MiniLM significantly outperforms TF-IDF across **every single metric**:
     - $\Delta$ MAE: **-0.0889** `[-0.1678, -0.0075]` ($p < 0.05$)
     - $\Delta$ RMSE: **-0.1913** `[-0.3444, -0.0487]` ($p < 0.05$)
     - $\Delta$ Pearson: **+0.3430** `[+0.1501, +0.5186]` ($p < 0.05$)
     - $\Delta$ Spearman: **+0.2709** `[+0.0841, +0.4430]` ($p < 0.05$)
     - $\Delta$ QWK: **+0.2370** `[+0.0766, +0.3957]` ($p < 0.05$)
   - None of the 95% confidence intervals cross zero, rigorously verifying that dense semantic embeddings provide substantial, statistically significant advantages over surface lexical matching.